In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# تخفيف الأخطاء باستخدام شبكات الموترات (TEM): دالة Qiskit من Algorithmiq
> **Note:** دوال Qiskit ميزة تجريبية متاحة فقط لمستخدمي IBM Quantum&reg; Premium Plan وFlex Plan وOn-Prem (عبر IBM Quantum Platform API). وهي في مرحلة إصدار معاينة وقابلة للتغيير.


<details>
<summary><b>إصدارات الحزم</b></summary>

تم تطوير الكود في هذه الصفحة باستخدام المتطلبات التالية.
نوصي باستخدام هذه الإصدارات أو أحدث منها.

```
qiskit[all]~=2.3.0
qiskit-ibm-catalog~=0.11.0
```
</details>
## نظرة عامة
طريقة تخفيف الأخطاء باستخدام شبكات الموترات (TEM) من Algorithmiq هي خوارزمية هجينة
كمّية-كلاسيكية مصمَّمة لتنفيذ تخفيف الضوضاء بالكامل في
مرحلة المعالجة اللاحقة الكلاسيكية. باستخدام TEM، يمكن للمستخدم حساب
القيم التوقعية للمراقِبات مع تخفيف الأخطاء الناجمة عن الضوضاء التي لا مفر منها
على الأجهزة الكمّية، وذلك بدقة أعلى وكفاءة أكبر في التكلفة،
مما يجعله خياراً بالغ الجاذبية لباحثي الكم وممارسي الصناعة على حدٍّ سواء.

تقوم الطريقة على بناء شبكة موترات تمثّل معكوس
قناة الضوضاء الكلية المؤثرة على حالة المعالج الكمّي، ثم
تطبيق الخريطة على نتائج القياسات المكتملة إعلامياً المكتسبة من
الحالة المضطربة للحصول على مقدِّرات غير متحيزة للمراقِبات.

كميزة إضافية، تستفيد TEM من القياسات المكتملة إعلامياً لإتاحة
الوصول إلى مجموعة واسعة من القيم التوقعية المخففة للمراقِبات، وتتمتع
بتكاليف أخذ عينات مثلى على الأجهزة الكمّية، كما هو موضَّح في Filippov et
al. (2023)، [arXiv:2307.11740](https://arxiv.org/abs/2307.11740)، وFilippov
et al. (2024)، [arXiv:2403.13542](https://arxiv.org/abs/2403.13542). يشير
تكلفة القياس إلى عدد القياسات الإضافية المطلوبة لإجراء تخفيف فعّال للأخطاء،
وهو عامل حاسم في جدوى الحوسبة الكمّية. لذا، يمتلك TEM القدرة على تحقيق
الميزة الكمّية في سيناريوهات معقدة، كالتطبيقات في مجالات الفوضى الكمّية،
وفيزياء الأجسام الكثيرة، وديناميكيات هوبارد، ومحاكاة كيمياء الجزيئات الصغيرة.

يمكن تلخيص الميزات والفوائد الرئيسية لـ TEM على النحو التالي:

1.  **تكلفة قياس مثلى**: TEM مثالية بالنسبة إلى
[الحدود النظرية](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.131.210601)،
بمعنى أنه لا توجد طريقة يمكنها تحقيق تكلفة قياس أصغر. بعبارة أخرى،
تتطلب TEM الحد الأدنى من القياسات الإضافية لإجراء تخفيف الأخطاء. وهذا بدوره
يعني أن TEM تستخدم الحد الأدنى من وقت التشغيل الكمّي.
2.  **فعالية التكلفة**: بما أن TEM تتعامل مع تخفيف الضوضاء بالكامل في
مرحلة المعالجة اللاحقة، لا حاجة لإضافة دوائر إضافية على الحاسوب
الكمّي، مما لا يجعل الحوسبة أرخص فحسب، بل يقلل أيضاً من
خطر إدخال أخطاء إضافية بسبب عيوب الأجهزة الكمّية.
3.  **تقدير مراقِبات متعددة**: بفضل القياسات المكتملة إعلامياً،
تقدّر TEM مراقِبات متعددة بكفاءة باستخدام بيانات القياس ذاتها من الحاسوب الكمّي.
4.  **تخفيف أخطاء القراءة**: تتضمن دالة TEM Qiskit أيضاً
[طريقة خاصة لتخفيف أخطاء القراءة](https://journals.aps.org/prresearch/abstract/10.1103/PhysRevResearch.5.033154)
قادرة على تقليص أخطاء القراءة بشكل ملحوظ بعد تشغيل معايرة قصير.
5.  **الدقة**: تُحسّن TEM بشكل ملحوظ دقة وموثوقية
المحاكاة الكمّية الرقمية، مما يجعل خوارزميات الكم أكثر دقة وموثوقية.
## الوصف
تتيح دالة TEM الحصول على قيم توقعية مخففة للأخطاء لمراقِبات
متعددة على دائرة كمّية بتكلفة أخذ عينات ضئيلة. تُقاس
الدائرة باستخدام قياس إيجابي لعوامل تشغيل قيّمة مكتملة إعلامياً (IC-POVM)،
وتُعالَج نتائج القياسات المجمّعة على حاسوب كلاسيكي. يُستخدم هذا القياس لتطبيق
طرق شبكة الموترات وبناء خريطة عكس الضوضاء. تطبّق الدالة خريطة تعكس
الدائرة المضطربة بالكامل باستخدام شبكات الموترات لتمثيل الطبقات المضطربة.

![مخطط TEM](../docs/images/guides/algorithmiq-tem/tem_scheme.svg "تقدير مخفَّف الأخطاء للمراقِب O عبر معالجة نتائج قياسات المعالج الكمّي المضطرب. U وN يدلان على عملية كمّية مثالية وخريطة الضوضاء المرتبطة بها، والتي يمكن أن تكون غير محلية بشكل عام (وتمتد إلى الصناديق الرمادية). D تمثّل موتر عوامل مزدوجة للتأثيرات في قياس IC. وحدة تخفيف الضوضاء M هي شبكة موترات تُقلَّص بكفاءة من المنتصف للخارج. تمثّل التكرار الأول للانكماش الخط الأرجواني المنقّط، والثاني الخط المتقطع، والثالث الخط المتصل.")

بمجرد إرسال الدوائر إلى الدالة، تُحوَّل وتُحسَّن لتقليل عدد الطبقات
بأبواب ثنائية Qubit (الأبواب الأكثر ضوضاءً في الأجهزة الكمّية). تُتعلَّم الضوضاء
المؤثرة على الطبقات من خلال
[Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/noise-learner-noise-learner)
باستخدام نموذج ضوضاء Pauli-Lindblad متفرق كما هو موضَّح في E. van den Berg, Z.
Minev, A. Kandala, K. Temme, Nat. Phys. (2023).
[arXiv:2201.09866](https://arxiv.org/abs/2201.09866).

نموذج الضوضاء وصف دقيق للضوضاء على الجهاز يلتقط الميزات الدقيقة،
بما فيها التداخل بين Qubits. غير أن الضوضاء على الأجهزة قد تتذبذب وتتغير،
وقد لا تكون الضوضاء المتعلَّمة دقيقة في نقطة إجراء التقدير، مما قد يؤدي إلى
نتائج غير دقيقة.
## البدء
قم بالمصادقة باستخدام [مفتاح IBM Quantum Platform API](http://quantum.cloud.ibm.com/)، وحدد دالة TEM على النحو التالي. (يفترض هذا المقتطف أنك قد [حفظت حسابك](/guides/functions#install-qiskit-functions-catalog-client) في بيئتك المحلية مسبقاً.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

tem_function_name = "algorithmiq/tem"

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Load your function
tem = catalog.load(tem_function_name)

## مثال
يوضح المقتطف التالي مثالاً يُستخدم فيه TEM لحساب القيم التوقعية لمراقِب ما في دائرة كمّية بسيطة.

In [2]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

# Create a quantum circuit
qc = QuantumCircuit(3)
qc.u(0.4, 0.9, -0.3, 0)
qc.u(-0.4, 0.2, 1.3, 1)
qc.u(-1.2, -1.2, 0.3, 2)
for _ in range(2):
    qc.barrier()
    qc.cx(0, 1)
    qc.cx(2, 1)
    qc.barrier()
    qc.u(0.4, 0.9, -0.3, 0)
    qc.u(-0.4, 0.2, 1.3, 1)
    qc.u(-1.2, -1.2, 0.3, 2)

# Define the observables
observable = SparsePauliOp("IYX", 1.0)

# Define the execution options
pub = (qc, [observable])
options = {"default_precision": 0.02}

# Define backend to use. TEM will choose the least-busy device reported by IBM if not specified
backend_name = "ibm_torino"

job = tem.run(pubs=[pub], backend_name=backend_name, options=options)

استخدم واجهات برمجة تطبيقات Qiskit Serverless للتحقق من حالة حِمل عمل دالة Qiskit:

In [3]:
print(job.status())

QUEUED


You can return the results as:

In [4]:
result = job.result()
evs = result[0].data.evs

يمكنك استرجاع النتائج على النحو التالي:

In [5]:
print(job.result())

PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(1,), dtype=float64>), stds=np.ndarray(<shape=(1,), dtype=float64>)), metadata={'evs_non_mitigated': array([-0.06314623]), 'stds_non_mitigated': array([0.05917147]), 'evs_mitigated_no_readout_mitigation': array([-0.06411205]), 'stds_mitigated_no_readout_mitigation': array([0.05992467]), 'evs_non_mitigated_with_readout_mitigation': array([-0.07028881]), 'stds_non_mitigated_with_readout_mitigation': array([0.06353934])})], metadata={'resource_usage': {'RUNNING: OPTIMIZING_FOR_HARDWARE': {'CPU_TIME': 0.915754}, 'RUNNING: WAITING_FOR_QPU': {'CPU_TIME': 18.804865}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 10.433445}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 159.0}}})


> **Info:** القيمة التوقعية للدائرة الخالية من الضوضاء للعامل المعطى يجب أن تكون حوالي `0.18409094298943401`.
## المُدخلات
**المعاملات**

الاسم | النوع | الوصف | مطلوب | الافتراضي | مثال
-- | -- | -- | -- | -- | --
`pubs` | Iterable[EstimatorPubLike] | كائنات تكرارية من نوع PUB (كتلة موحدة بدائية)، مثل الصفوف `(circuit, observables)` أو `(circuit, observables, parameters, precision)`. راجع [نظرة عامة على PUBs](/guides/primitive-input-output#overview-of-pubs) لمزيد من المعلومات. إذا تم تمرير دائرة غير ISA، ستُحوَّل بالإعدادات المثلى. إذا تم تمرير دائرة ISA، لن تُحوَّل؛ في هذه الحالة يجب تعريف المراقِب على كامل QPU. | نعم | لا ينطبق | (circuit, observables)
`backend_name` | str | اسم الـ Backend لإجراء الاستعلام.| لا | إذا لم يُحدَّد، سيُستخدم الـ Backend الأقل انشغالاً. | "ibm_fez"
`options` | dict | خيارات الإدخال. راجع قسم `Options` لمزيد من التفاصيل. | لا | راجع قسم `Options` لمزيد من التفاصيل.| {"max_bond_dimension": 100\}

> **Caution:** يوجد حالياً لـ TEM القيود التالية:
> 
>   - الدوائر ذات المعاملات غير مدعومة. يجب ضبط وسيطة parameters على `None` إذا تم تحديد الدقة. ستُزال هذه القيود في الإصدارات المستقبلية.
>   - تُدعم فقط الدوائر غير المحتوية على حلقات. ستُزال هذه القيود في الإصدارات المستقبلية.
>   - الأبواب غير الوحدوية، كالإعادة والقياس وكافة أشكال تدفق التحكم، غير مدعومة. سيُضاف دعم الإعادة في الإصدارات القادمة.
### الخيارات
قاموس يحتوي على الخيارات المتقدمة لـ TEM. قد يحتوي القاموس على المفاتيح الموضحة في الجدول التالي. إذا لم يُقدَّم أي من الخيارات، سيُستخدم القيمة الافتراضية المدرجة في الجدول. القيم الافتراضية مناسبة للاستخدام النموذجي لـ TEM.

الاسم | الاختيارات | الوصف | الافتراضي
-- | -- | -- | --
`tem_max_bond_dimension` | int | أقصى بُعد رابطة يُستخدم لشبكات الموترات. | 500 |
`tem_compression_cutoff` | float | قيمة الحد للضغط المستخدمة لشبكات الموترات. | 1e-16
`compute_shadows_bias_from_observable` | bool | علامة بوليانية تشير إلى ما إذا كان ينبغي تصميم التحيز لبروتوكول قياس الظلال الكلاسيكية وفقاً للمراقِب PUB أم لا. إذا كانت False، سيُستخدم بروتوكول الظلال الكلاسيكية (احتمال متساوٍ لقياس Z وX وY).| False
`shadows_bias` | np.ndarray | التحيز المستخدم لبروتوكول قياس الظلال الكلاسيكية العشوائية، مصفوفة أحادية أو ثنائية الأبعاد بحجم 3 أو شكل (num_qubits, 3) على التوالي. الترتيب هو ZXY | np.array([1 / 3, 1 / 3, 1 / 3])
`max_execution_time` | int أو `None` | أقصى وقت تنفيذ على QPU بالثواني. إذا تجاوز وقت التشغيل هذه القيمة، سيُلغى الوظيفة. إذا كانت `None`، سيُطبَّق الحد الافتراضي المحدد من Qiskit Runtime. | `None`
`num_randomizations` | int | عدد العشوائيات المستخدمة لتعلم الضوضاء وتدوير البوابات. | 32
`max_layers_to_learn` | int | العدد الأقصى للطبقات الفريدة المراد تعلمها. | 4
`mitigate_readout_error` | bool | علامة بوليانية تشير إلى ما إذا كان ينبغي إجراء تخفيف أخطاء القراءة أم لا. | True
`num_readout_calibration_shots` | int | عدد اللقطات المستخدمة لتخفيف أخطاء القراءة. | 10000
`default_precision` | float | الدقة الافتراضية المستخدمة للـ PUBs التي لم تُحدَّد لها دقة. |0.02
`seed` | int أو `None` | تعيين بذرة مولّد الأرقام العشوائية لإمكانية إعادة الإنتاج. إذا كانت `None`، لا تُعيَّن البذرة. | None
## المُخرجات
[PrimitiveResults](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) من Qiskit يحتوي على النتيجة المخففة بواسطة TEM. تُعاد نتيجة كل PUB كـ[PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) يحتوي على الحقول التالية:

الاسم | النوع | الوصف
-- | -- | --
data | DataBin | [DataBin](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin) من Qiskit يحتوي على المراقِب المخفَّف بواسطة TEM وخطأه المعياري. يحتوي DataBin على الحقول التالية: <ul><li>`evs`: قيمة المراقِب المخففة بواسطة TEM.</li><li>`stds`: الخطأ المعياري للمراقِب المخفَّف بواسطة TEM.</li></ul>
metadata | dict | قاموس يحتوي على نتائج إضافية. يحتوي القاموس على المفاتيح التالية: <ul><li>`"evs_non_mitigated"`: قيمة المراقِب بدون تخفيف الأخطاء.</li><li>`"stds_non_mitigated"`: الخطأ المعياري للنتيجة بدون تخفيف الأخطاء.</li><li>`"evs_mitigated_no_readout_mitigation"`: قيمة المراقِب مع تخفيف الأخطاء لكن بدون تخفيف أخطاء القراءة.</li><li>`"stds_mitigated_no_readout_mitigation"`: الخطأ المعياري للنتيجة مع تخفيف الأخطاء لكن بدون تخفيف أخطاء القراءة.</li><li>`"evs_non_mitigated_with_readout_mitigation"`: قيمة المراقِب بدون تخفيف الأخطاء لكن مع تخفيف أخطاء القراءة.</li><li>`"stds_non_mitigated_with_readout_mitigation"`: الخطأ المعياري للنتيجة بدون تخفيف الأخطاء لكن مع تخفيف أخطاء القراءة.</li></ul>
## استرجاع رسائل الخطأ
إذا كانت حالة حِمل العمل ERROR، استخدم job.result() لاسترجاع رسالة الخطأ على النحو التالي: